# Kumbh-Reunite — Evaluation

Retrieval (Top-k, mAP, CMC) + verification (ROC/AUC, EER, P/R/F1) on a
gallery/query split, with optional 'Digital Kumbh' augmentation of queries
to measure robustness. Swap `('sample', SAMPLE)` for a benchmark like
`('lfw', None)` once a dataset is placed under `datasets/`.

In [ ]:
import os, sys, json
REPO = os.path.abspath('../..')
sys.path.insert(0, REPO)
from face_retrieval.config import load_config, get_logger, set_seed
from face_retrieval.pipeline import SearchPipeline, collect_samples

cfg = load_config()
set_seed(int(cfg.project.seed))
log = get_logger('kumbh', cfg.logging.level)
SAMPLE = os.path.join(REPO, 'backend', '_testdata', 'faces')
SOURCE, PATH = 'sample', SAMPLE   # e.g. ('lfw', None)

In [ ]:
samples = collect_samples(cfg, SOURCE, PATH, limit=None)
pipe = SearchPipeline(cfg, log)
metrics, art = pipe.evaluate(samples, query_per_id=1, augment_queries=True)
print(json.dumps({k: v for k, v in metrics['retrieval'].items() if k != 'cmc'}, indent=2))
print(json.dumps({k: metrics['verification'].get(k) for k in
      ('auc','eer','precision','recall','f1','genuine_mean','impostor_mean')}, indent=2))
print('timing:', metrics['timing'])

In [ ]:
# Curves + plots
from face_retrieval.modules import visualization as viz
from IPython.display import Image, display
o = cfg.paths.output_dir
viz.plot_cmc(metrics['retrieval']['cmc'], os.path.join(o, 'cmc.png'))
if 'roc' in metrics['verification']:
    viz.plot_roc(metrics['verification']['roc'], metrics['verification']['auc'], os.path.join(o, 'roc.png'))
viz.plot_similarity_hist(art['sim'], art['same'], os.path.join(o, 'sim_hist.png'))
viz.plot_embedding_scatter(pipe.gallery_embs, pipe.gallery_ids, os.path.join(o, 'embeddings.png'),
                           method=cfg.evaluation.embedding_plot)
for name in ['cmc.png', 'roc.png', 'sim_hist.png', 'embeddings.png']:
    p = os.path.join(o, name)
    if os.path.exists(p):
        display(Image(p))